# Gemini and Claude comparison

Compare the recorded study and dataset tables.

Method and inputs: [workflow guide](../../docs/workflow.md).

In [2]:
from __future__ import annotations

import sys
from pathlib import Path

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "notebooks" / "_shared.py").is_file():
        sys.path.insert(0, str(_parent / "notebooks"))
        break
else:
    raise FileNotFoundError("Start Jupyter inside the repository.")


from _shared import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index,
)

load_environment()


import json
from pathlib import Path
from typing import Any

import pandas as pd


def find_project_root() -> Path:
    required_path = Path("data") / "results" / "review_final.csv"

    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / required_path).exists():
            return candidate

    raise FileNotFoundError(
        "Could not find data/results/review_final.csv "
        "from the current working directory."
    )


def normalize_doi(value: Any) -> str | None:
    """Normalize a DOI and remove common URL/prefix variants."""
    if value is None or pd.isna(value):
        return None

    doi = str(value).strip().lower()

    prefixes = (
        "https://doi.org/",
        "http://doi.org/",
        "https://dx.doi.org/",
        "http://dx.doi.org/",
        "doi:",
    )

    for prefix in prefixes:
        if doi.startswith(prefix):
            doi = doi[len(prefix):].strip()
            break

    return doi or None


def normalized_dois_from_dataframe(df: pd.DataFrame) -> set[str]:
    if "DOI" not in df.columns:
        raise KeyError(
            f"Missing DOI column. Available columns: {list(df.columns)}"
        )

    return {
        doi
        for value in df["DOI"]
        if (doi := normalize_doi(value)) is not None
    }


def extract_dois_from_json(data: Any) -> set[str]:
    """
    Recursively extract DOI values from common JSON structures.

    Supported examples:
    - {"10.xxxx/...": ["Dataset A", "Dataset B"]}
    - [{"DOI": "10.xxxx/...", "datasets": [...]}]
    - {"studies": [{"doi": "10.xxxx/..."}]}
    """
    dois: set[str] = set()

    if isinstance(data, dict):
        for key, value in data.items():
            normalized_key = normalize_doi(key)

            # Handles JSON objects whose keys are directly DOI strings.
            if normalized_key and normalized_key.startswith("10."):
                dois.add(normalized_key)

            # Handles fields named DOI or doi.
            if key.strip().lower() == "doi":
                normalized_value = normalize_doi(value)
                if normalized_value:
                    dois.add(normalized_value)
            else:
                dois.update(extract_dois_from_json(value))

    elif isinstance(data, list):
        for item in data:
            dois.update(extract_dois_from_json(item))

    return dois


def print_doi_list(title: str, dois: set[str] | list[str]) -> None:
    sorted_dois = sorted(dois)

    print(f"\n{title} ({len(sorted_dois)}):")

    if not sorted_dois:
        print("  None")
        return

    for doi in sorted_dois:
        print(f"  - {doi}")


root = find_project_root()

gemini_path = Path(results_directory()) / "review_final.csv"
claude_path = Path(results_directory()) / "CLAUDE_review_final.csv"
study_datasets_path = root / "data" / "study_datasets.json"

if not claude_path.exists():
    raise FileNotFoundError(f"Missing file: {claude_path}")

if not study_datasets_path.exists():
    raise FileNotFoundError(f"Missing file: {study_datasets_path}")


gemini_df = pd.read_csv(gemini_path)
claude_df = pd.read_csv(claude_path)

with study_datasets_path.open("r", encoding="utf-8") as file:
    study_datasets_data = json.load(file)


gemini_dois = normalized_dois_from_dataframe(gemini_df)
claude_dois = normalized_dois_from_dataframe(claude_df)
study_datasets_dois = extract_dois_from_json(study_datasets_data)


# Pairwise comparisons
gemini_claude_common = gemini_dois & claude_dois
gemini_json_common = gemini_dois & study_datasets_dois
claude_json_common = claude_dois & study_datasets_dois

gemini_only_vs_claude = gemini_dois - claude_dois
claude_only_vs_gemini = claude_dois - gemini_dois

gemini_only_vs_json = gemini_dois - study_datasets_dois
json_only_vs_gemini = study_datasets_dois - gemini_dois

claude_only_vs_json = claude_dois - study_datasets_dois
json_only_vs_claude = study_datasets_dois - claude_dois


# Three-way comparison
common_to_all = gemini_dois & claude_dois & study_datasets_dois

only_gemini = gemini_dois - claude_dois - study_datasets_dois
only_claude = claude_dois - gemini_dois - study_datasets_dois
only_json = study_datasets_dois - gemini_dois - claude_dois

gemini_and_claude_only = (
    gemini_dois & claude_dois
) - study_datasets_dois

gemini_and_json_only = (
    gemini_dois & study_datasets_dois
) - claude_dois

claude_and_json_only = (
    claude_dois & study_datasets_dois
) - gemini_dois


print("DOI counts")
print(f"  Gemini:             {len(gemini_dois)}")
print(f"  Claude:             {len(claude_dois)}")
print(f"  study_datasets.json: {len(study_datasets_dois)}")


print("\nPAIRWISE COMPARISON: GEMINI VS CLAUDE")
print_doi_list("DOIs in common", gemini_claude_common)
print_doi_list(
    "DOIs only in Gemini review_final.csv",
    gemini_only_vs_claude,
)
print_doi_list(
    "DOIs only in Claude CLAUDE_review_final.csv",
    claude_only_vs_gemini,
)


print("\nPAIRWISE COMPARISON: GEMINI VS study_datasets.json")
print_doi_list(
    "DOIs in both Gemini and study_datasets.json",
    gemini_json_common,
)
print_doi_list(
    "DOIs only in Gemini",
    gemini_only_vs_json,
)
print_doi_list(
    "DOIs only in study_datasets.json",
    json_only_vs_gemini,
)


print("\nPAIRWISE COMPARISON: CLAUDE VS study_datasets.json")
print_doi_list(
    "DOIs in both Claude and study_datasets.json",
    claude_json_common,
)
print_doi_list(
    "DOIs only in Claude",
    claude_only_vs_json,
)
print_doi_list(
    "DOIs only in study_datasets.json",
    json_only_vs_claude,
)


print("\nTHREE-WAY COMPARISON")
print_doi_list(
    "DOIs common to Gemini, Claude, and study_datasets.json",
    common_to_all,
)
print_doi_list(
    "DOIs only in Gemini",
    only_gemini,
)
print_doi_list(
    "DOIs only in Claude",
    only_claude,
)
print_doi_list(
    "DOIs only in study_datasets.json",
    only_json,
)
print_doi_list(
    "DOIs in Gemini and Claude, but not in study_datasets.json",
    gemini_and_claude_only,
)
print_doi_list(
    "DOIs in Gemini and study_datasets.json, but not in Claude",
    gemini_and_json_only,
)
print_doi_list(
    "DOIs in Claude and study_datasets.json, but not in Gemini",
    claude_and_json_only,
)


DOI counts
  Gemini:             91
  Claude:             87
  study_datasets.json: 94

PAIRWISE COMPARISON: GEMINI VS CLAUDE

DOIs in common (84):
  - 10.1007/978-3-030-58577-8_14
  - 10.1007/978-3-031-04881-4_46
  - 10.1007/978-3-031-18913-5_17
  - 10.1007/978-3-031-19772-7_25
  - 10.1007/978-3-031-19772-7_27
  - 10.1007/978-3-031-19839-7_7
  - 10.1007/978-3-031-44216-2_19
  - 10.1007/978-3-031-61678-5_6
  - 10.1007/978-3-319-10599-4_36
  - 10.1007/s00521-023-09068-w
  - 10.1007/s10489-023-04613-5
  - 10.1007/s10489-023-05166-3
  - 10.1007/s10489-024-05349-6
  - 10.1007/s11263-021-01486-4
  - 10.1007/s11263-022-01695-5
  - 10.1007/s11760-021-01890-w
  - 10.1007/s40747-022-00892-6
  - 10.1016/j.compbiomed.2024.108382
  - 10.1016/j.ins.2024.120347
  - 10.1016/j.jvcir.2021.103304
  - 10.1016/j.jvcir.2022.103625
  - 10.1016/j.knosys.2021.107388
  - 10.1016/j.patcog.2017.12.007
  - 10.1016/j.patcog.2021.108095
  - 10.1016/j.patrec.2022.04.015
  - 10.1038/s41597-022-01188-7
  - 10.1109/acc

In [4]:
root = find_project_root()
gemini_path = Path(results_directory()) / "review.csv"
claude_path = Path(results_directory()) / "CLAUDE_review.csv"

gemini_df = pd.read_csv(gemini_path)
claude_df = pd.read_csv(claude_path)

def normalized_dois(df: pd.DataFrame) -> set[str]:
    if "DOI" not in df.columns:
        raise KeyError(f"Missing DOI column. Available columns: {list(df.columns)}")
    return set(
        df["DOI"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )


gemini_dois = normalized_dois(gemini_df)
claude_dois = normalized_dois(claude_df)

common_dois = sorted(gemini_dois & claude_dois)
gemini_only_dois = sorted(gemini_dois - claude_dois)
claude_only_dois = sorted(claude_dois - gemini_dois)

def print_doi_list(title: str, dois: list[str]) -> None:
    print(f"\n{title} ({len(dois)}):")
    if not dois:
        print("  None")
        return
    for doi in dois:
        print(f"  - {doi}")


print(f"Gemini DOI count: {len(gemini_dois)}")
print(f"Claude DOI count: {len(claude_dois)}")

print_doi_list("DOIs in common", common_dois)
print_doi_list("DOIs only in Gemini review.csv", gemini_only_dois)
print_doi_list("DOIs only in Claude CLAUDE_review.csv", claude_only_dois)

Gemini DOI count: 75
Claude DOI count: 66

DOIs in common (66):
  - 10.1007/978-3-030-58577-8_14
  - 10.1007/978-3-031-04881-4_46
  - 10.1007/978-3-031-18913-5_17
  - 10.1007/978-3-031-19772-7_27
  - 10.1007/978-3-031-19839-7_7
  - 10.1007/978-3-031-44216-2_19
  - 10.1007/978-3-319-10599-4_36
  - 10.1007/s00521-023-09068-w
  - 10.1007/s10489-023-04613-5
  - 10.1007/s10489-023-05166-3
  - 10.1007/s10489-024-05349-6
  - 10.1007/s11263-022-01695-5
  - 10.1007/s11760-021-01890-w
  - 10.1007/s40747-022-00892-6
  - 10.1016/j.compbiomed.2024.108382
  - 10.1016/j.ins.2024.120347
  - 10.1016/j.jvcir.2021.103304
  - 10.1016/j.jvcir.2022.103625
  - 10.1016/j.knosys.2021.107388
  - 10.1016/j.patcog.2017.12.007
  - 10.1016/j.patcog.2021.108095
  - 10.1016/j.patrec.2022.04.015
  - 10.1038/s41597-022-01188-7
  - 10.1109/access.2024.3423462
  - 10.1109/cbms61543.2024.00016
  - 10.1109/cvpr.2019.00039
  - 10.1109/cvpr42600.2020.00986
  - 10.1109/cvpr52688.2022.00296
  - 10.1109/cvpr52729.2023.00238
  -

In [5]:
root = find_project_root()
gemini_path = Path(results_directory()) / "df_evaluated_validation_abstract.csv"
claude_path = Path(results_directory()) / "CLAUDE_df_evaluated_validation_abstract.csv"

gemini_df = pd.read_csv(gemini_path)
claude_df = pd.read_csv(claude_path)

def normalized_dois(df: pd.DataFrame) -> set[str]:
    if "DOI" not in df.columns:
        raise KeyError(f"Missing DOI column. Available columns: {list(df.columns)}")
    return set(
        df["DOI"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )


gemini_dois = normalized_dois(gemini_df)
claude_dois = normalized_dois(claude_df)

common_dois = sorted(gemini_dois & claude_dois)
gemini_only_dois = sorted(gemini_dois - claude_dois)
claude_only_dois = sorted(claude_dois - gemini_dois)

def print_doi_list(title: str, dois: list[str]) -> None:
    print(f"\n{title} ({len(dois)}):")
    if not dois:
        print("  None")
        return
    for doi in dois:
        print(f"  - {doi}")


print(f"Gemini DOI count: {len(gemini_dois)}")
print(f"Claude DOI count: {len(claude_dois)}")

print_doi_list("DOIs in common", common_dois)
print_doi_list("DOIs only in Gemini df_evaluated_validation_abstract.csv", gemini_only_dois)
print_doi_list("DOIs only in Claude CLAUDE_df_evaluated_validation_abstract.csv", claude_only_dois)

Gemini DOI count: 21
Claude DOI count: 29

DOIs in common (20):
  - 10.1007/978-3-030-00767-6_12
  - 10.1007/978-3-030-20876-9_10
  - 10.1007/978-3-031-19772-7_25
  - 10.1007/s11263-021-01486-4
  - 10.1109/access.2023.3305372
  - 10.1109/access.2023.3316009
  - 10.1109/cvpr52688.2022.00323
  - 10.1109/cvprw.2017.16
  - 10.1109/icassp48485.2024.10447069
  - 10.1109/icct.2015.7399879
  - 10.1109/icip42928.2021.9506257
  - 10.1109/icmew46912.2020.9106049
  - 10.1109/icmlc58545.2023.10327994
  - 10.1109/tcsvt.2020.3017727
  - 10.1109/tip.2024.3362135
  - 10.1109/tmm.2023.3294800
  - 10.1109/wacv48630.2021.00044
  - 10.1145/3650400.3650642
  - 10.1155/2022/8950732
  - 10.26555/ijain.v9i1.919

DOIs only in Gemini df_evaluated_validation_abstract.csv (1):
  - 10.1109/tvcg.2023.3326524

DOIs only in Claude CLAUDE_df_evaluated_validation_abstract.csv (9):
  - 10.1007/978-3-031-61678-5_6
  - 10.1007/s10489-022-03984-5
  - 10.1109/icaace61206.2024.10548995
  - 10.1109/icip.2018.8451364
  - 10.110

In [7]:
root = find_project_root()
gemini_path = Path(results_directory()) / "df_screening_passed.csv"
claude_path = Path(results_directory()) / "CLAUDE_df_screening_passed.csv"

gemini_df = pd.read_csv(gemini_path)
claude_df = pd.read_csv(claude_path)

def normalized_dois(df: pd.DataFrame) -> set[str]:
    if "DOI" not in df.columns:
        raise KeyError(f"Missing DOI column. Available columns: {list(df.columns)}")
    return set(
        df["DOI"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.lower()
    )


gemini_dois = normalized_dois(gemini_df)
claude_dois = normalized_dois(claude_df)

common_dois = sorted(gemini_dois & claude_dois)
gemini_only_dois = sorted(gemini_dois - claude_dois)
claude_only_dois = sorted(claude_dois - gemini_dois)

def print_doi_list(title: str, dois: list[str]) -> None:
    print(f"\n{title} ({len(dois)}):")
    if not dois:
        print("  None")
        return
    for doi in dois:
        print(f"  - {doi}")


print(f"Gemini DOI count: {len(gemini_dois)}")
print(f"Claude DOI count: {len(claude_dois)}")

print_doi_list("DOIs in common", common_dois)
print_doi_list("DOIs only in Gemini df_screening_passed.csv", gemini_only_dois)
print_doi_list("DOIs only in Claude CLAUDE_df_screening_passed.csv", claude_only_dois)

Gemini DOI count: 96
Claude DOI count: 95

DOIs in common (94):
  - 10.1007/978-3-030-00767-6_12
  - 10.1007/978-3-030-20876-9_10
  - 10.1007/978-3-030-58577-8_14
  - 10.1007/978-3-031-04881-4_46
  - 10.1007/978-3-031-18913-5_17
  - 10.1007/978-3-031-19772-7_25
  - 10.1007/978-3-031-19772-7_27
  - 10.1007/978-3-031-19839-7_7
  - 10.1007/978-3-031-44216-2_19
  - 10.1007/978-3-031-61678-5_6
  - 10.1007/978-3-319-10599-4_36
  - 10.1007/s00521-023-09068-w
  - 10.1007/s10489-022-03984-5
  - 10.1007/s10489-023-04613-5
  - 10.1007/s10489-023-05166-3
  - 10.1007/s10489-024-05349-6
  - 10.1007/s11263-021-01486-4
  - 10.1007/s11263-022-01695-5
  - 10.1007/s11760-021-01890-w
  - 10.1007/s40747-022-00892-6
  - 10.1016/j.compbiomed.2024.108382
  - 10.1016/j.ins.2024.120347
  - 10.1016/j.jvcir.2021.103304
  - 10.1016/j.jvcir.2022.103625
  - 10.1016/j.knosys.2021.107388
  - 10.1016/j.patcog.2017.12.007
  - 10.1016/j.patcog.2021.108095
  - 10.1016/j.patrec.2022.04.015
  - 10.1038/s41597-022-01188-7
  